# 📄 W3: 납품 문서 자동화
**hexa-1 — Week 3** | ⏱️ 60분 | 🎯 납품확인서+이메일+검수보고 자동 생성

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 납품 정보 입력 (✏️)

In [ ]:
DOC={
    'supplier':'✏️ [납품사명]','buyer':'✏️ [발주사명]',
    'item':'✏️ [품목명]','qty':100,'unit_price':50000,
    'delivery_date':'2026-04-01','contract_no':'2026-001'
}
DOC['total']=DOC['qty']*DOC['unit_price']
print(f'✅ 납품 총액: {DOC["total"]:,}원')


## Step 2: 납품확인서 자동 생성

In [ ]:
p1=f"""아래 정보로 공식 납품확인서를 작성하세요.
납품사:{DOC['supplier']} / 발주사:{DOC['buyer']} / 품목:{DOC['item']}
수량:{DOC['qty']}개 / 단가:{DOC['unit_price']:,}원 / 총액:{DOC['total']:,}원
납기일:{DOC['delivery_date']} / 계약번호:{DOC['contract_no']}
보기 좋은 마크다운 형식으로."""
confirm=model.generate_content(p1)
print(confirm.text)


## Step 3: 납품 이메일 초안

In [ ]:
p2=f"""납품 완료 이메일을 정중하고 전문적으로 작성하세요.
발신:{DOC['supplier']} / 수신:{DOC['buyer']} / 품목:{DOC['item']} {DOC['qty']}개
형식: 제목/인사/본문/첨부파일 안내/서명 포함."""
email=model.generate_content(p2)
print(email.text)


## Step 4: 검수보고서 생성

In [ ]:
p3=f"""납품 검수보고서를 공식 문서 형식으로 작성하세요.
품목:{DOC['item']} {DOC['qty']}개 / 납품일:{DOC['delivery_date']}
검수 항목: 수량확인/외관검사/기능검사/포장상태/합격판정 포함."""
report=model.generate_content(p3)
print(report.text)


## Step 5: ZIP 다운로드

In [ ]:
import zipfile
for name,content in [('납품확인서.md',confirm.text),('납품이메일.md',email.text),('검수보고서.md',report.text)]:
    with open(name,'w',encoding='utf-8') as f: f.write(content)
with zipfile.ZipFile('delivery_docs.zip','w') as z:
    for name in ['납품확인서.md','납품이메일.md','검수보고서.md']: z.write(name)
from google.colab import files; files.download('delivery_docs.zip')
print('✅ 완료!')
